In [1]:
# Preparación: localizar las fuentes y convertir las seis tablas Markdown a DataFrames.
from pathlib import Path
import json
import math
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
def formato_float(x):
    return f'{x:.3e}' if x != 0 and abs(x) < 1e-4 else f'{x:,.6f}'

pd.set_option('display.float_format', formato_float)

def localizar_base():
    cwd = Path.cwd().resolve()
    candidatos = []
    for raiz in (cwd, *cwd.parents):
        candidatos.append(raiz)
        candidatos.append(raiz / 'diplomado-ml-seguros' / 'Modulo_4' / 'evaluacion_M4T2')
    for base in candidatos:
        if (base / 'evaluacion_M4T2.md').is_file():
            return base
    raise FileNotFoundError('No se encontró evaluacion_M4T2.md desde el directorio actual.')

def celdas_markdown(linea):
    return [celda.strip() for celda in linea.strip().strip('|').split('|')]

def es_separador_markdown(linea):
    celdas = celdas_markdown(linea)
    return bool(celdas) and all(re.fullmatch(r':?-{3,}:?', celda) for celda in celdas)

def extraer_tablas_markdown(texto):
    lineas = texto.splitlines()
    tablas = []
    i = 0
    while i < len(lineas) - 1:
        if lineas[i].lstrip().startswith('|') and es_separador_markdown(lineas[i + 1]):
            columnas = celdas_markdown(lineas[i])
            filas = []
            i += 2
            while i < len(lineas) and lineas[i].lstrip().startswith('|'):
                fila = celdas_markdown(lineas[i])
                if len(fila) == len(columnas):
                    filas.append(fila)
                i += 1
            tablas.append(pd.DataFrame(filas, columns=columnas))
        else:
            i += 1
    return tablas

def a_numero(valor):
    texto = str(valor).strip().replace(',', '').replace('−', '-').replace('%', '')
    if texto in {'', '—', '-'}:
        return np.nan
    coincidencia = re.search(r'[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?', texto)
    if not coincidencia:
        raise ValueError(f'No se pudo convertir a número: {valor!r}')
    return float(coincidencia.group())

BASE = localizar_base()
RUTA_EVALUACION = BASE / 'evaluacion_M4T2.md'
RUTA_METADATOS = BASE / 'documentacion' / 'modelos' / 'metadatos.json'
texto_evaluacion = RUTA_EVALUACION.read_text(encoding='utf-8')
metadatos = json.loads(RUTA_METADATOS.read_text(encoding='utf-8'))
coincidencia_variable = re.search(r'\*\*Variable asignada:\*\*\s*`([^`]+)`', texto_evaluacion)
variable_asignada = coincidencia_variable.group(1)
assert variable_asignada == 'antiguedad_vehiculo_cat'
assert variable_asignada in metadatos['frecuencia']['formula']
assert variable_asignada in metadatos['severidad']['formula']
tablas = extraer_tablas_markdown(texto_evaluacion)
assert len(tablas) == 6, f'Se esperaban 6 tablas y se encontraron {len(tablas)}.'

diagnostico, rf_frec, modelos, rf_sev, validacion, tarifa = [tabla.copy() for tabla in tablas]
diagnostico.columns = ['metrica', 'valor']
rf_frec.columns = ['nivel', 'RF_frec', 'IC_inf', 'IC_sup', 'p', 'tasa_emp']
modelos.columns = ['modelo', 'AIC', 'BIC', 'pseudoR2_McF']
rf_sev.columns = ['nivel', 'RF_sev', 'severidad_emp']
validacion.columns = ['metrica', 'valor', 'ideal']
tarifa.columns = ['nivel', 'prima_pura_modelo', 'factor_tarifa']

for columna in ['valor']:
    diagnostico[columna] = diagnostico[columna].map(a_numero)
for columna in ['RF_frec', 'IC_inf', 'IC_sup', 'p', 'tasa_emp']:
    rf_frec[columna] = rf_frec[columna].map(a_numero)
for columna in ['AIC', 'BIC', 'pseudoR2_McF']:
    modelos[columna] = modelos[columna].map(a_numero)
for columna in ['RF_sev', 'severidad_emp']:
    rf_sev[columna] = rf_sev[columna].map(a_numero)
validacion['valor'] = validacion['valor'].map(a_numero)
for columna in ['prima_pura_modelo', 'factor_tarifa']:
    tarifa[columna] = tarifa[columna].map(a_numero)

resumen_fuentes = pd.DataFrame({
    'fuente': [str(RUTA_EVALUACION), str(RUTA_METADATOS)],
    'uso': ['tablas oficiales P1–P9', 'familia, liga, offset y ponderadores']
})
display(resumen_fuentes)
print(f'Tablas extraídas: {len(tablas)}')
print('Variable asignada:', variable_asignada)
print('Frecuencia:', metadatos['frecuencia']['family'], '| offset:', metadatos['frecuencia']['offset'])
print('Severidad:', metadatos['severidad']['family'], '| weights:', metadatos['severidad']['weights'])

,fuente,uso
0,C:\Users\vic_m\OneDrive\Desktop\Victor\Diploma...,tablas oficiales P1–P9
1,C:\Users\vic_m\OneDrive\Desktop\Victor\Diploma...,"familia, liga, offset y ponderadores"


Tablas extraídas: 6
Variable asignada: antiguedad_vehiculo_cat
Frecuencia: Poisson | offset: log(exposicion)
Severidad: Gamma(log) | weights: num_siniestros


In [2]:
# P1: diagnóstico de equidispersión y familia sugerida.
def valor_diagnostico(fragmento):
    mascara = diagnostico['metrica'].str.contains(fragmento, case=False, regex=False)
    return diagnostico.loc[mascara, 'valor'].iloc[0]

phi_pearson = valor_diagnostico('Pearson')
alpha_ct = valor_diagnostico('Cameron-Trivedi')
z_ct = valor_diagnostico('z')
p_ct = valor_diagnostico('p-value')
hay_sobredispersion = phi_pearson > 1
ct_rechaza_equidispersion = p_ct < 0.05
sobredispersion_leve = hay_sobredispersion and ct_rechaza_equidispersion and phi_pearson < 1.5
if not ct_rechaza_equidispersion:
    familia_sugerida = 'Poisson'
elif sobredispersion_leve:
    familia_sugerida = 'Quasi-Poisson'
elif phi_pearson > 2:
    familia_sugerida = 'Binomial Negativa'
else:
    familia_sugerida = 'comparar Quasi-Poisson y Binomial Negativa'
factor_ajuste_ee = math.sqrt(phi_pearson)

resultado_p1 = pd.Series({
    'phi_Pearson': phi_pearson,
    'alpha_Cameron_Trivedi': alpha_ct,
    'z': z_ct,
    'p_value': p_ct,
    'rechaza_equidispersion': hay_sobredispersion and ct_rechaza_equidispersion,
    'intensidad': 'leve' if sobredispersion_leve else 'no leve',
    'familia_sugerida': familia_sugerida,
    'multiplicador_errores_estandar': factor_ajuste_ee
}, name='P1')
display(resultado_p1.to_frame('resultado'))
assert resultado_p1['rechaza_equidispersion']
assert familia_sugerida == 'Quasi-Poisson'

,resultado
phi_Pearson,1.166400
alpha_Cameron_Trivedi,0.074400
z,15.800000
p_value,3.700e-56
rechaza_equidispersion,True
intensidad,leve
familia_sugerida,Quasi-Poisson
multiplicador_errores_estandar,1.080000


In [3]:
# P2: extremos de frecuencia, porcentajes, intervalos de confianza y significancia.
frec_calc = rf_frec.copy()
frec_calc['es_referencia'] = frec_calc['nivel'].str.contains('(ref)', regex=False)
frec_calc['variacion_pct_vs_ref'] = 100 * (frec_calc['RF_frec'] - 1)
frec_calc['IC_cruza_1'] = ((frec_calc['IC_inf'] <= 1) & (frec_calc['IC_sup'] >= 1)).astype('boolean')
frec_calc['significativo_5pct'] = (frec_calc['p'] < 0.05).astype('boolean')
frec_calc.loc[frec_calc['es_referencia'], ['IC_cruza_1', 'significativo_5pct']] = pd.NA

referencia_frec = frec_calc.loc[frec_calc['es_referencia']].iloc[0]
no_referencia_frec = frec_calc.loc[~frec_calc['es_referencia']]
max_no_ref_frec = no_referencia_frec.loc[no_referencia_frec['RF_frec'].idxmax()]
min_frec = frec_calc.loc[frec_calc['RF_frec'].idxmin()]
hay_ic_que_cruza = bool(no_referencia_frec['IC_cruza_1'].any())
hay_p_mayor_005 = bool((no_referencia_frec['p'] > 0.05).any())
decision_niveles = 'mantener todos' if not hay_ic_que_cruza and not hay_p_mayor_005 else 'revisar/combinar'

display(frec_calc[['nivel', 'RF_frec', 'variacion_pct_vs_ref', 'IC_inf', 'IC_sup', 'p', 'IC_cruza_1', 'significativo_5pct']])
print(f"Máximo global: {referencia_frec['nivel']} | RF={referencia_frec['RF_frec']:.4f} | cambio={referencia_frec['variacion_pct_vs_ref']:.2f}%")
print(f"Máximo no referencia: {max_no_ref_frec['nivel']} | RF={max_no_ref_frec['RF_frec']:.4f} | cambio={max_no_ref_frec['variacion_pct_vs_ref']:.2f}%")
print(f"Mínimo: {min_frec['nivel']} | RF={min_frec['RF_frec']:.4f} | cambio={min_frec['variacion_pct_vs_ref']:.2f}%")
print('Decisión:', decision_niveles)
assert max_no_ref_frec['nivel'] == '(10, 15]'
assert min_frec['nivel'] == '(15, 50]'
assert decision_niveles == 'mantener todos'

,nivel,RF_frec,variacion_pct_vs_ref,IC_inf,IC_sup,p,IC_cruza_1,significativo_5pct
0,"(-1, 1] (ref)",1.000000,0.000000,1.000000,1.000000,0.000000,<NA>,<NA>
1,"(1, 2]",0.760000,-24.000000,0.703700,0.820700,0.000000,False,True
2,"(10, 15]",0.882500,-11.750000,0.826000,0.942800,0.000200,False,True
3,"(15, 50]",0.679500,-32.050000,0.611300,0.755300,0.000000,False,True
4,"(2, 3]",0.725400,-27.460000,0.670300,0.785000,0.000000,False,True
5,"(3, 4]",0.792300,-20.770000,0.733600,0.855700,0.000000,False,True
6,"(4, 5]",0.793400,-20.660000,0.734400,0.857300,0.000000,False,True
7,"(5, 10]",0.821600,-17.840000,0.771800,0.874600,0.000000,False,True


Máximo global: (-1, 1] (ref) | RF=1.0000 | cambio=0.00%
Máximo no referencia: (10, 15] | RF=0.8825 | cambio=-11.75%
Mínimo: (15, 50] | RF=0.6795 | cambio=-32.05%
Decisión: mantener todos


In [4]:
# P3: comprobación numérica del puente entre el GLM one-way y la tasa empírica.
# Con liga log y offset: exp(beta_0 + beta_nivel) = suma(siniestros) / suma(exposición).
tasa_base = float(referencia_frec['tasa_emp'])
puente_p3 = frec_calc[['nivel', 'RF_frec', 'tasa_emp']].copy()
puente_p3['tasa_reconstruida_base_x_RF'] = tasa_base * puente_p3['RF_frec']
puente_p3['diferencia_abs'] = (puente_p3['tasa_reconstruida_base_x_RF'] - puente_p3['tasa_emp']).abs()
max_diferencia_redondeada = puente_p3['diferencia_abs'].max()
coincidencia_oficial = re.search(r'diferencia máx\s*=\s*([0-9.eE+-]+)', texto_evaluacion)
max_diferencia_oficial = float(coincidencia_oficial.group(1))

display(puente_p3)
print(f'Diferencia máxima con valores redondeados de la tabla: {max_diferencia_redondeada:.8f}')
print(f'Diferencia máxima oficial con precisión interna: {max_diferencia_oficial:.8f}')
print('La discrepancia entre ambas diferencias proviene del redondeo de RF y tasas mostrado en el enunciado.')
print('Aportes adicionales del GLM: efectos multiplicativos multivariados, IC, pruebas de hipótesis y predicción.')
assert max_diferencia_redondeada < 1e-4
assert math.isclose(max_diferencia_oficial, 5.2e-5, rel_tol=0, abs_tol=1e-12)

,nivel,RF_frec,tasa_emp,tasa_reconstruida_base_x_RF,diferencia_abs
0,"(-1, 1] (ref)",1.000000,0.169800,0.169800,0.000000
1,"(1, 2]",0.760000,0.129000,0.129048,4.800e-05
2,"(10, 15]",0.882500,0.149800,0.149848,4.850e-05
3,"(15, 50]",0.679500,0.115400,0.115379,2.090e-05
4,"(2, 3]",0.725400,0.123100,0.123173,7.292e-05
5,"(3, 4]",0.792300,0.134500,0.134533,3.254e-05
6,"(4, 5]",0.793400,0.134700,0.134719,1.932e-05
7,"(5, 10]",0.821600,0.139500,0.139508,7.680e-06


Diferencia máxima con valores redondeados de la tabla: 0.00007292
Diferencia máxima oficial con precisión interna: 0.00005200
La discrepancia entre ambas diferencias proviene del redondeo de RF y tasas mostrado en el enunciado.
Aportes adicionales del GLM: efectos multiplicativos multivariados, IC, pruebas de hipótesis y predicción.


In [5]:
# P4: demostración de CV constante en Gamma y del sesgo al retransformar log(Y).
config_severidad = pd.Series(metadatos['severidad'], name='configuracion_severidad')
display(config_severidad.to_frame())

phi_gamma_demo = 0.36
medias_demo = np.array([500.0, 1000.0, 2000.0, 4000.0])
varianzas_gamma = phi_gamma_demo * medias_demo**2
desv_gamma = np.sqrt(varianzas_gamma)
cv_gamma = desv_gamma / medias_demo
demostracion_gamma = pd.DataFrame({
    'media_mu': medias_demo,
    'varianza_phi_por_mu2': varianzas_gamma,
    'desviacion': desv_gamma,
    'CV': cv_gamma
})
display(demostracion_gamma)
assert np.allclose(cv_gamma, math.sqrt(phi_gamma_demo))

media_log_demo = 7.0
sigma_log_demo = 0.7
retransformacion_ingenua = math.exp(media_log_demo)
media_lognormal = math.exp(media_log_demo + sigma_log_demo**2 / 2)
factor_correccion = media_lognormal / retransformacion_ingenua
print(f'exp(E[log Y]) = {retransformacion_ingenua:,.2f}')
print(f'E[Y] lognormal = exp(E[log Y] + sigma²/2) = {media_lognormal:,.2f}')
print(f'Factor de corrección por retransformación = {factor_correccion:.4f}')
print('Gamma(log) modela E[Y|X] directamente; OLS sobre log(Y) modela E[log(Y)|X].')
print('Lognormal no es un GLM estándar sobre Y: su estadístico suficiente involucra log(y), no y.')

,configuracion_severidad
respuesta,severidad
formula,severidad ~ C(edad_conductor_cat) + C(antigued...
family,Gamma(log)
weights,num_siniestros
filtro,num_siniestros>0


,media_mu,varianza_phi_por_mu2,desviacion,CV
0,500.000000,"90,000.000000",300.000000,0.600000
1,"1,000.000000","360,000.000000",600.000000,0.600000
2,"2,000.000000","1,440,000.000000","1,200.000000",0.600000
3,"4,000.000000","5,760,000.000000","2,400.000000",0.600000


exp(E[log Y]) = 1,096.63
E[Y] lognormal = exp(E[log Y] + sigma²/2) = 1,401.08
Factor de corrección por retransformación = 1.2776
Gamma(log) modela E[Y|X] directamente; OLS sobre log(Y) modela E[log(Y)|X].
Lognormal no es un GLM estándar sobre Y: su estadístico suficiente involucra log(y), no y.


In [6]:
# P5: selección por AIC/BIC y comparación de pseudo R².
modelos_calc = modelos.copy()
modelos_calc['delta_AIC_vs_mejor'] = modelos_calc['AIC'] - modelos_calc['AIC'].min()
modelos_calc['delta_BIC_vs_mejor'] = modelos_calc['BIC'] - modelos_calc['BIC'].min()
ganador_aic = modelos_calc.loc[modelos_calc['AIC'].idxmin()]
ganador_bic = modelos_calc.loc[modelos_calc['BIC'].idxmin()]
poisson = modelos_calc.loc[modelos_calc['modelo'].eq('Poisson')].iloc[0]
binomial_negativa = modelos_calc.loc[modelos_calc['modelo'].eq('Binomial Negativa')].iloc[0]
reduccion_aic = poisson['AIC'] - binomial_negativa['AIC']
reduccion_bic = poisson['BIC'] - binomial_negativa['BIC']
mejora_pseudo_r2 = binomial_negativa['pseudoR2_McF'] - poisson['pseudoR2_McF']

display(modelos_calc)
print(f"Ganador AIC: {ganador_aic['modelo']} | reducción frente a Poisson: {reduccion_aic:.1f}")
print(f"Ganador BIC: {ganador_bic['modelo']} | reducción frente a Poisson: {reduccion_bic:.1f}")
print(f'Mejora absoluta del pseudo R² de McFadden: {mejora_pseudo_r2:.4f}')
print('Un pseudo R² bajo es esperable con la aleatoriedad irreducible de la frecuencia; se usa comparativamente, junto con validación.')
print('P1 usa Quasi-Poisson como corrección inferencial; P5 elige Binomial Negativa entre modelos con verosimilitud mediante AIC/BIC.')
assert ganador_aic['modelo'] == ganador_bic['modelo'] == 'Binomial Negativa'
assert math.isclose(reduccion_aic, 156.0, abs_tol=1e-9)
assert math.isclose(reduccion_bic, 155.9, abs_tol=1e-9)

,modelo,AIC,BIC,pseudoR2_McF,delta_AIC_vs_mejor,delta_BIC_vs_mejor
0,Poisson,"125,081.700000","125,261.700000",0.019800,156.000000,155.900000
1,Binomial Negativa,"124,925.700000","125,105.800000",0.021000,0.000000,0.000000


Ganador AIC: Binomial Negativa | reducción frente a Poisson: 156.0
Ganador BIC: Binomial Negativa | reducción frente a Poisson: 155.9
Mejora absoluta del pseudo R² de McFadden: 0.0012
Un pseudo R² bajo es esperable con la aleatoriedad irreducible de la frecuencia; se usa comparativamente, junto con validación.
P1 usa Quasi-Poisson como corrección inferencial; P5 elige Binomial Negativa entre modelos con verosimilitud mediante AIC/BIC.


In [7]:
# P6: comparación de dirección, magnitud y orden de los RF de frecuencia y severidad.
comparacion_rf = frec_calc[['nivel', 'RF_frec']].merge(rf_sev[['nivel', 'RF_sev']], on='nivel', validate='one_to_one')
comparacion_rf['variacion_frec_pct'] = 100 * (comparacion_rf['RF_frec'] - 1)
comparacion_rf['variacion_sev_pct'] = 100 * (comparacion_rf['RF_sev'] - 1)
comparacion_rf['misma_direccion'] = np.sign(comparacion_rf['RF_frec'] - 1) == np.sign(comparacion_rf['RF_sev'] - 1)
comparacion_rf['RF_producto_frec_x_sev'] = comparacion_rf['RF_frec'] * comparacion_rf['RF_sev']
es_ref_comparacion = comparacion_rf['nivel'].str.contains('(ref)', regex=False)
todos_misma_direccion = bool(comparacion_rf.loc[~es_ref_comparacion, 'misma_direccion'].all())
minimo_frecuencia = comparacion_rf.loc[comparacion_rf['RF_frec'].idxmin()]
minimo_severidad = comparacion_rf.loc[comparacion_rf['RF_sev'].idxmin()]

display(comparacion_rf)
print('Todos los niveles no referencia apuntan en la misma dirección:', todos_misma_direccion)
print(f"Mínimo de frecuencia: {minimo_frecuencia['nivel']} | RF={minimo_frecuencia['RF_frec']:.4f} | RF_sev={minimo_frecuencia['RF_sev']:.4f}")
print(f"Mínimo de severidad: {minimo_severidad['nivel']} | RF={minimo_severidad['RF_sev']:.4f} | RF_frec={minimo_severidad['RF_frec']:.4f}")
print('Como el orden y la magnitud difieren, Frecuencia y Severidad deben modelarse por separado y luego multiplicarse.')
assert todos_misma_direccion
assert minimo_frecuencia['nivel'] == '(15, 50]'
assert minimo_severidad['nivel'] == '(3, 4]'

,nivel,RF_frec,RF_sev,variacion_frec_pct,variacion_sev_pct,misma_direccion,RF_producto_frec_x_sev
0,"(-1, 1] (ref)",1.000000,1.000000,0.000000,0.000000,True,1.000000
1,"(1, 2]",0.760000,0.714100,-24.000000,-28.590000,True,0.542716
2,"(10, 15]",0.882500,0.740000,-11.750000,-26.000000,True,0.653050
3,"(15, 50]",0.679500,0.909900,-32.050000,-9.010000,True,0.618277
4,"(2, 3]",0.725400,0.686500,-27.460000,-31.350000,True,0.497987
5,"(3, 4]",0.792300,0.627500,-20.770000,-37.250000,True,0.497168
6,"(4, 5]",0.793400,0.734100,-20.660000,-26.590000,True,0.582435
7,"(5, 10]",0.821600,0.734200,-17.840000,-26.580000,True,0.603219


Todos los niveles no referencia apuntan en la misma dirección: True
Mínimo de frecuencia: (15, 50] | RF=0.6795 | RF_sev=0.9099
Mínimo de severidad: (3, 4] | RF=0.6275 | RF_frec=0.7923
Como el orden y la magnitud difieren, Frecuencia y Severidad deben modelarse por separado y luego multiplicarse.


In [8]:
# P7: calibración (ratio predicho/observado) y discriminación (Gini).
fila_gini = validacion.loc[validacion['metrica'].str.contains('Gini', case=False)].iloc[0]
fila_ratio = validacion.loc[validacion['metrica'].str.contains('Ratio', case=False)].iloc[0]
gini_test = float(fila_gini['valor'])
ratio_pred_obs = float(fila_ratio['valor'])
umbral_gini = a_numero(fila_gini['ideal'])
desviacion_calibracion_pct = 100 * (ratio_pred_obs - 1)
tolerancia_didactica_calibracion_pct = 5.0
bien_calibrado = abs(desviacion_calibracion_pct) <= tolerancia_didactica_calibracion_pct
discriminacion = 'aceptable' if gini_test > umbral_gini else 'modesta/limitada'

resultado_p7 = pd.Series({
    'ratio_pred_obs': ratio_pred_obs,
    'desviacion_pred_vs_obs_pct': desviacion_calibracion_pct,
    'criterio_didactico_calibracion': f'|desviación| <= {tolerancia_didactica_calibracion_pct:.1f}%',
    'bien_calibrado_en_promedio': bien_calibrado,
    'Gini_test': gini_test,
    'umbral_Gini_del_enunciado': umbral_gini,
    'discriminacion': discriminacion
}, name='P7')
display(resultado_p7.to_frame('resultado'))
print('Calibración mide coincidencia del nivel agregado; Gini mide el ordenamiento de riesgos.')
assert math.isclose(desviacion_calibracion_pct, 2.49, abs_tol=1e-9)
assert bien_calibrado and discriminacion == 'modesta/limitada'

,resultado
ratio_pred_obs,1.024900
desviacion_pred_vs_obs_pct,2.490000
criterio_didactico_calibracion,|desviación| <= 5.0%
bien_calibrado_en_promedio,True
Gini_test,0.231500
umbral_Gini_del_enunciado,0.300000
discriminacion,modesta/limitada


Calibración mide coincidencia del nivel agregado; Gini mide el ordenamiento de riesgos.


In [9]:
# P8: prima pura máxima/mínima y recargo o descuento frente al promedio.
tarifa_calc = tarifa.copy()
tarifa_calc['variacion_pct_vs_promedio'] = 100 * (tarifa_calc['factor_tarifa'] - 1)
tarifa_calc['prima_promedio_implicita'] = tarifa_calc['prima_pura_modelo'] / tarifa_calc['factor_tarifa']
max_prima = tarifa_calc.loc[tarifa_calc['prima_pura_modelo'].idxmax()]
min_prima = tarifa_calc.loc[tarifa_calc['prima_pura_modelo'].idxmin()]
prima_promedio_estimada = tarifa_calc['prima_promedio_implicita'].mean()

display(tarifa_calc)
print(f'Prima promedio implícita por los factores: {prima_promedio_estimada:.2f} unidades monetarias')
print(f"Máxima: {max_prima['nivel']} | prima={max_prima['prima_pura_modelo']:.2f} | factor={max_prima['factor_tarifa']:.4f} | cambio={max_prima['variacion_pct_vs_promedio']:.2f}%")
print(f"Mínima: {min_prima['nivel']} | prima={min_prima['prima_pura_modelo']:.2f} | factor={min_prima['factor_tarifa']:.4f} | cambio={min_prima['variacion_pct_vs_promedio']:.2f}%")
assert max_prima['nivel'] == '(-1, 1] (ref)'
assert min_prima['nivel'] == '(2, 3]'
assert math.isclose(max_prima['variacion_pct_vs_promedio'], 67.38, abs_tol=1e-9)
assert math.isclose(min_prima['variacion_pct_vs_promedio'], -18.60, abs_tol=1e-9)

,nivel,prima_pura_modelo,factor_tarifa,variacion_pct_vs_promedio,prima_promedio_implicita
0,"(-1, 1] (ref)",305.200000,1.673800,67.380000,182.339587
1,"(1, 2]",163.760000,0.898100,-10.190000,182.340497
2,"(10, 15]",194.410000,1.066200,6.620000,182.339148
3,"(15, 50]",186.660000,1.023700,2.370000,182.338576
4,"(2, 3]",148.410000,0.814000,-18.600000,182.321867
5,"(3, 4]",150.550000,0.825600,-17.440000,182.352229
6,"(4, 5]",176.120000,0.965900,-3.410000,182.337716
7,"(5, 10]",180.370000,0.989200,-1.080000,182.339264


Prima promedio implícita por los factores: 182.34 unidades monetarias
Máxima: (-1, 1] (ref) | prima=305.20 | factor=1.6738 | cambio=67.38%
Mínima: (2, 3] | prima=148.41 | factor=0.8140 | cambio=-18.60%


In [10]:
# P9: generación programática de una conclusión técnica con frecuencia, severidad, IC y prima pura.
base_p9 = (
    frec_calc[['nivel', 'RF_frec', 'IC_inf', 'IC_sup']]
    .merge(rf_sev[['nivel', 'RF_sev']], on='nivel', validate='one_to_one')
    .merge(tarifa_calc[['nivel', 'prima_pura_modelo', 'factor_tarifa', 'variacion_pct_vs_promedio']], on='nivel', validate='one_to_one')
)
fila_objetivo = base_p9.loc[base_p9['prima_pura_modelo'].idxmin()]
fila_referencia = base_p9.loc[base_p9['nivel'].str.contains('(ref)', regex=False)].iloc[0]
no_ref_p9 = base_p9.loc[~base_p9['nivel'].str.contains('(ref)', regex=False)]
niveles_con_doble_descuento = int(((no_ref_p9['RF_frec'] < 1) & (no_ref_p9['RF_sev'] < 1)).sum())
total_niveles_no_ref = len(no_ref_p9)

lineas_nota_tecnica = [
    f'La antigüedad se asocia con menor frecuencia y severidad respecto de vehículos de hasta un año en {niveles_con_doble_descuento} de {total_niveles_no_ref} niveles no referencia.',
    f"Para {fila_objetivo['nivel']}, RF_frec={fila_objetivo['RF_frec']:.4f} (IC95% {fila_objetivo['IC_inf']:.4f}–{fila_objetivo['IC_sup']:.4f}) y RF_sev={fila_objetivo['RF_sev']:.4f}.",
    f"Su prima pura es {fila_objetivo['prima_pura_modelo']:.2f} y queda {abs(fila_objetivo['variacion_pct_vs_promedio']):.2f}% debajo del promedio.",
    f"La referencia alcanza {fila_referencia['prima_pura_modelo']:.2f} ({fila_referencia['variacion_pct_vs_promedio']:+.2f}%); se conserva la segmentación y se monitorea fuera de muestra."
]
print('\n'.join(lineas_nota_tecnica))
assert niveles_con_doble_descuento == total_niveles_no_ref == 7
assert fila_objetivo['nivel'] == '(2, 3]'
assert math.isclose(fila_objetivo['RF_frec'], 0.7254, abs_tol=1e-12)
assert math.isclose(fila_objetivo['RF_sev'], 0.6865, abs_tol=1e-12)

La antigüedad se asocia con menor frecuencia y severidad respecto de vehículos de hasta un año en 7 de 7 niveles no referencia.
Para (2, 3], RF_frec=0.7254 (IC95% 0.6703–0.7850) y RF_sev=0.6865.
Su prima pura es 148.41 y queda 18.60% debajo del promedio.
La referencia alcanza 305.20 (+67.38%); se conserva la segmentación y se monitorea fuera de muestra.


In [11]:
# Validación integral de los resultados usados en las nueve respuestas.
controles = {
    'variable_asignada_correcta': variable_asignada == 'antiguedad_vehiculo_cat',
    'P1_sobredispersion_leve_y_QuasiPoisson': sobredispersion_leve and familia_sugerida == 'Quasi-Poisson',
    'P2_extremos_y_significancia': max_no_ref_frec['nivel'] == '(10, 15]' and min_frec['nivel'] == '(15, 50]' and decision_niveles == 'mantener todos',
    'P3_reproduce_tasas_dentro_del_redondeo': max_diferencia_redondeada < 1e-4,
    'P4_CV_Gamma_constante': bool(np.allclose(cv_gamma, math.sqrt(phi_gamma_demo))),
    'P5_Binomial_Negativa_gana_AIC_BIC': ganador_aic['modelo'] == ganador_bic['modelo'] == 'Binomial Negativa',
    'P6_misma_direccion_distinto_orden': todos_misma_direccion and minimo_frecuencia['nivel'] != minimo_severidad['nivel'],
    'P7_calibrado_discriminacion_modesta': bien_calibrado and discriminacion == 'modesta/limitada',
    'P8_extremos_tarifa_correctos': max_prima['nivel'] == '(-1, 1] (ref)' and min_prima['nivel'] == '(2, 3]',
    'P9_nota_integra_frecuencia_severidad_tarifa': len(lineas_nota_tecnica) == 4
}
display(pd.Series(controles, name='cumple').to_frame())
assert all(controles.values())
print('Validación completada: P1–P9 reproducidas sin errores.')

,cumple
variable_asignada_correcta,True
P1_sobredispersion_leve_y_QuasiPoisson,True
P2_extremos_y_significancia,True
P3_reproduce_tasas_dentro_del_redondeo,True
P4_CV_Gamma_constante,True
P5_Binomial_Negativa_gana_AIC_BIC,True
P6_misma_direccion_distinto_orden,True
P7_calibrado_discriminacion_modesta,True
P8_extremos_tarifa_correctos,True
P9_nota_integra_frecuencia_severidad_tarifa,True


Validación completada: P1–P9 reproducidas sin errores.
